In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SQL Practice") \
    .getOrCreate()

In [16]:
df = spark.read.csv("orders_dataset.csv", header=True, inferSchema=True)
df1 = spark.read.csv("products_dataset.csv", header=True, inferSchema=True)
df2 = spark.read.csv("customers_dataset.csv", header=True, inferSchema=True)

df.createOrReplaceTempView("orders")
df1.createOrReplaceTempView("products")
df2.createOrReplaceTempView("customers")

In [17]:
#Customers with multiple category orders

spark.sql("""
SELECT c.customer_id, c.customer_name, COUNT(DISTINCT p.category) AS category_count
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN products p ON o.product_id = p.product_id
GROUP BY c.customer_id, c.customer_name
HAVING COUNT(DISTINCT p.category) > 1
""").show()

+-----------+--------------------+--------------+
|customer_id|       customer_name|category_count|
+-----------+--------------------+--------------+
|         59|          Jim Deleon|             2|
|         76|        Pamela Price|             4|
|         86|  Elizabeth Sheppard|             2|
|        117|Miss Melody Mclau...|             5|
|        118|        John Johnson|             5|
|         73|        Lauren Ortiz|             3|
|        107|         David Brown|             4|
|         63|     Michael Bridges|             3|
|          6|    Theresa Ferguson|             5|
|         28|       Joshua Hardin|             2|
|         96|          Gina Ortiz|             5|
|          8|       Michael Sloan|             3|
|        110|        Jeremy Munoz|             3|
|        111|         Shawn Walsh|             3|
|         20|        Paul Stevens|             3|
|         21|        Jordan Jones|             2|
|         30|    Katherine Hansen|             2|


In [18]:
#detect duplicate transactions

spark.sql("""
SELECT order_id, COUNT(*) AS count
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1
""").show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [19]:
#top selling product in each state. 

spark.sql("""
SELECT state, product_name, order_count
FROM (
    SELECT 
        c.state,
        p.product_name,
        COUNT(*) AS order_count,
        ROW_NUMBER() OVER (
            PARTITION BY c.state 
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON o.product_id = p.product_id
    GROUP BY c.state, p.product_name
) t
WHERE rn = 1
ORDER BY state
""").show()

+-----------+---------------+-----------+
|      state|   product_name|order_count|
+-----------+---------------+-----------+
|      Delhi|  Place Product|          8|
|  Karnataka|  Place Product|          5|
|Maharashtra|College Product|          5|
| Tamil Nadu| Return Product|          7|
|  Telangana|  Stock Product|          4|
+-----------+---------------+-----------+



In [20]:
# customer segmentaion (VIP for 100000+ spends, Platinum for 50000 - 100000, others regular)

spark.sql("""
SELECT 
    c.customer_id, 
    c.customer_name,
    SUM(o.sales_amount) AS total_spent,
    CASE 
        WHEN SUM(o.sales_amount) >= 100000 THEN 'VIP'
        WHEN SUM(o.sales_amount) >= 50000 THEN 'Platinum'
        ELSE 'Regular'
    END AS segment
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_spent DESC
""").show()

+-----------+-----------------+------------------+-------+
|customer_id|    customer_name|       total_spent|segment|
+-----------+-----------------+------------------+-------+
|         93| Shirley Harrison|         845556.64|    VIP|
|          6| Theresa Ferguson|         720304.25|    VIP|
|         11|    Darrell Jones| 592001.0900000001|    VIP|
|         45|     Jason Nelson|         591651.49|    VIP|
|         80|     Jason Garcia|         588679.86|    VIP|
|        106|     Shane Miller|          581160.2|    VIP|
|         52|  Jennifer Hunter|         565407.64|    VIP|
|         56|    Matthew Evans|         532683.28|    VIP|
|         64|      Nicole Meza|         503299.37|    VIP|
|        104|  Alison Mcdonald|         497268.72|    VIP|
|         78|Joshua Washington| 494396.5199999999|    VIP|
|         48| Kristine Schmidt|         493638.62|    VIP|
|         94|     Sheila Stone| 493296.8900000001|    VIP|
|        114|    Alan Anderson|         489207.55|    VI

In [21]:
#detect suspicious high speed orders

spark.sql("""
SELECT customer_id, COUNT(*) AS order_count, MIN(order_date) AS first_order, MAX(order_date) AS last_order
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 10 AND DATEDIFF(MAX(order_date), MIN(order_date)) < 1
""").show()

+-----------+-----------+-----------+----------+
|customer_id|order_count|first_order|last_order|
+-----------+-----------+-----------+----------+
+-----------+-----------+-----------+----------+



In [22]:
#identify products commonly bought together. logic is same date purchase by same customer.

spark.sql("""
SELECT
    o1.product_id AS product_id_1,
    o2.product_id AS product_id_2,
    COUNT(*) AS co_purchase_count
FROM orders o1
JOIN orders o2 ON o1.customer_id = o2.customer_id
    AND o1.order_date = o2.order_date
    AND o1.product_id < o2.product_id
GROUP BY o1.product_id, o2.product_id
ORDER BY co_purchase_count DESC
LIMIT 20
""").show()

+------------+------------+-----------------+
|product_id_1|product_id_2|co_purchase_count|
+------------+------------+-----------------+
|          22|          33|                1|
|           3|          10|                1|
|           4|           6|                1|
+------------+------------+-----------------+

